# Merge, Clean, and Store the Seven Steam CSV Datasets in MySQL

**Purpose:** Combine the seven daily CSV files, perform transparent cleaning, save a cleaned CSV backup, and load the result into MySQL.

**Before running:** Extract `data.zip` so this notebook is in the same project folder as the `data` directory. Install packages with `%pip install pandas sqlalchemy pymysql`.

**Important cleaning decisions**
- The seven files have the same columns, so we append their rows.
- Repeated `app_id` values are retained because a game can appear in multiple categories or on multiple collection dates.
- Exact duplicate records are removed only if the entire record matches.
- `discount_percent` is dropped if it is empty in every file.
- `Free` prices become `0.0`; currency prices such as `$19.99` become numeric USD values.
- Missing price/review values are not guessed. Missing original prices can be legitimate when a game is not discounted.

## Cell 1: Import packages

In [1]:
from pathlib import Path
import pandas as pd

## Cell 2: Find and load the seven CSV files

In [2]:
DATA_FOLDER = Path("data")
csv_files = sorted(DATA_FOLDER.glob("*.csv"))

if len(csv_files) != 7:
    raise ValueError(
        f"Expected 7 CSV files in {DATA_FOLDER.resolve()}, but found {len(csv_files)}. "
        "Check that the seven original CSV files are in the data folder."
    )

dataframes = []
for file_path in csv_files:
    df = pd.read_csv(file_path, encoding="utf-8-sig")
    df["source_file"] = file_path.name
    dataframes.append(df)

print("Files loaded:", len(dataframes))
for file_path, df in zip(csv_files, dataframes):
    print(f"{file_path.name}: {len(df):,} rows and {len(df.columns)-1} original columns")

Files loaded: 7
steam_top_sellers_2026-10-14.csv: 500 rows and 11 original columns
steam_top_sellers_2026-10-15.csv: 500 rows and 11 original columns
steam_top_sellers_2026-10-16.csv: 500 rows and 11 original columns
steam_top_sellers_2026-10-17.csv: 377 rows and 11 original columns
steam_top_sellers_2026-10-18.csv: 500 rows and 11 original columns
steam_top_sellers_2026-10-19.csv: 500 rows and 11 original columns
steam_top_sellers_2026_10_13.csv: 500 rows and 11 original columns


## Cell 3: Check that the column names match

In [3]:
expected_columns = list(dataframes[0].columns.drop("source_file"))
for file_path, df in zip(csv_files, dataframes):
    current_columns = list(df.columns.drop("source_file"))
    if set(current_columns) != set(expected_columns):
        raise ValueError(f"Column mismatch in {file_path.name}")

print("All seven files contain the same original columns.")
print(expected_columns)

All seven files contain the same original columns.
['app_id', 'rank', 'category', 'title', 'price', 'original_price', 'discount_percent', 'release_date', 'review_summary', 'platforms', 'scraped_at']


## Cell 4: Merge by appending rows

In [4]:
merged = pd.concat(dataframes, ignore_index=True)
print("Merged shape:", merged.shape)
display(merged.head())

Merged shape: (3377, 12)


,app_id,rank,category,title,price,original_price,discount_percent,release_date,review_summary,platforms,scraped_at,source_file
0,570,1,Action,Dota 2,Free,NaN,NaN,9-Jul-13,"Very Positive 87% of the 841,501 user reviews ...","win, mac, linux",14/10/2026 0:26,steam_top_sellers_2026-10-14.csv
1,3393110,2,Action,AION 2,Free,NaN,NaN,5-Oct-26,"Mixed 54% of the 14,022 user reviews for this ...",win,14/10/2026 0:26,steam_top_sellers_2026-10-14.csv
2,730,3,Action,Counter-Strike 2,Free,NaN,NaN,21-Aug-12,"Very Positive 85% of the 2,632,865 user review...","win, linux",14/10/2026 0:26,steam_top_sellers_2026-10-14.csv
3,2828710,4,Action,Valor Mortis,$38.23,$44.98,NaN,13-Oct-26,Very Positive 83% of the 67 user reviews for t...,win,14/10/2026 0:26,steam_top_sellers_2026-10-14.csv
4,2807960,5,Action,Battlefield™ 6,$27.99,$69.99,NaN,10-Oct-25,"Mixed 65% of the 156,196 user reviews for this...",win,14/10/2026 0:26,steam_top_sellers_2026-10-14.csv


## Cell 5: Inspect data quality before cleaning

In [5]:
print("Missing values per column:")
display(merged.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

print("Exact duplicate rows:", merged.drop(columns=["source_file"]).duplicated().sum())
print("Unique app IDs:", merged["app_id"].nunique())
print("Rows per source file:")
display(merged["source_file"].value_counts().to_frame("rows"))

Missing values per column:


,missing_values
discount_percent,3377
original_price,2074
review_summary,103
price,9
title,0
category,0
rank,0
app_id,0
release_date,0
platforms,0


Exact duplicate rows: 0
Unique app IDs: 691
Rows per source file:


,rows
source_file,
steam_top_sellers_2026-10-14.csv,500
steam_top_sellers_2026-10-15.csv,500
steam_top_sellers_2026-10-16.csv,500
steam_top_sellers_2026-10-18.csv,500
steam_top_sellers_2026-10-19.csv,500
steam_top_sellers_2026_10_13.csv,500
steam_top_sellers_2026-10-17.csv,377


## Cell 6: Clean column names and text

In [6]:
cleaned = merged.copy()
cleaned.columns = [c.strip().lower().replace(" ", "_") for c in cleaned.columns]

for column in cleaned.select_dtypes(include="object").columns:
    cleaned[column] = cleaned[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )
    cleaned[column] = cleaned[column].replace("", pd.NA)

# Remove discount_percent only when it is completely empty in all records.
if "discount_percent" in cleaned.columns and cleaned["discount_percent"].isna().all():
    cleaned = cleaned.drop(columns=["discount_percent"])
    print("Dropped discount_percent because it was empty in every source file.")

Dropped discount_percent because it was empty in every source file.


C:\Users\User\AppData\Local\Temp\ipykernel_1868\1809522117.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in cleaned.select_dtypes(include="object").columns:


## Cell 7: Convert price columns into numeric USD values

In [7]:
def price_to_number(value):
    """Convert a Steam price such as '$19.99' or 'Free' into a number."""
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if text.lower() == "free":
        return 0.0

    text = text.replace("$", "").replace(",", "").strip()
    try:
        return float(text)
    except ValueError:
        return pd.NA

# Convert current and original prices into numeric USD columns.
if "price" in cleaned.columns:
    cleaned["price_usd"] = cleaned["price"].map(price_to_number).astype("Float64")
    cleaned = cleaned.drop(columns=["price"])

if "original_price" in cleaned.columns:
    cleaned["original_price_usd"] = cleaned["original_price"].map(price_to_number).astype("Float64")
    cleaned = cleaned.drop(columns=["original_price"])

# Calculate discount only when both prices exist and original price is greater than zero.
# If either price is missing, or original price is zero, leave discount_percent missing.
valid_discount = (
    cleaned["price_usd"].notna()
    & cleaned["original_price_usd"].notna()
    & (cleaned["original_price_usd"] > 0)
)

cleaned["discount_percent"] = pd.Series(pd.NA, index=cleaned.index, dtype="Float64")
cleaned.loc[valid_discount, "discount_percent"] = (
    (cleaned.loc[valid_discount, "original_price_usd"]
     - cleaned.loc[valid_discount, "price_usd"])
    / cleaned.loc[valid_discount, "original_price_usd"]
    * 100
).round(2)

print("Discount percentages calculated:", int(valid_discount.sum()))
print("Discount percentages left missing:", int((~valid_discount).sum()))
display(cleaned[["price_usd", "original_price_usd", "discount_percent"]].head(10))


Discount percentages calculated: 1303
Discount percentages left missing: 2074


,price_usd,original_price_usd,discount_percent
0,0.0,<NA>,<NA>
1,0.0,<NA>,<NA>
2,0.0,<NA>,<NA>
3,38.23,44.98,15.01
4,27.99,69.99,60.01
5,19.99,39.99,50.01
6,0.0,<NA>,<NA>
7,5.39,5.99,10.02
8,19.99,<NA>,<NA>
9,24.99,<NA>,<NA>


## Cell 8: Convert dates and remove exact duplicate records

In [8]:


# Convert release_date into a proper date format.
cleaned["release_date"] = pd.to_datetime(
    cleaned["release_date"],
    errors="coerce",
    format="mixed",
    dayfirst=True
)

# Convert scraped_at while supporting different timestamp formats.
cleaned["scraped_at"] = pd.to_datetime(
    cleaned["scraped_at"],
    errors="coerce",
    format="mixed",
    dayfirst=True
)

# Check whether the timestamps were preserved.
print("Total records:", len(cleaned))

print(
    "Missing scraped_at values:",
    cleaned["scraped_at"].isna().sum()
)

print(
    "Available scraped_at values:",
    cleaned["scraped_at"].notna().sum()
)

# Display sample timestamps to confirm the results.
display(cleaned[["source_file", "scraped_at"]].head(10))



Total records: 3377
Missing scraped_at values: 0
Available scraped_at values: 3377


,source_file,scraped_at
0,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
1,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
2,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
3,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
4,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
5,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
6,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
7,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
8,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00
9,steam_top_sellers_2026-10-14.csv,2026-10-14 00:26:00


## Cell 9: Validate the cleaned dataset

In [9]:
print("Missing values after cleaning:")
display(cleaned.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

print("Rows by category:")
display(cleaned["category"].value_counts(dropna=False).to_frame("rows"))

print("Sample of cleaned records:")
display(cleaned.head())

Missing values after cleaning:


,missing_values
discount_percent,2074
original_price_usd,2074
review_summary,103
price_usd,9
title,0
category,0
rank,0
app_id,0
scraped_at,0
platforms,0


Rows by category:


,rows
category,
Action,1176
Adventure,1162
Singleplayer,1039


Sample of cleaned records:


,app_id,rank,category,title,release_date,review_summary,platforms,scraped_at,source_file,price_usd,original_price_usd,discount_percent
0,570,1,Action,Dota 2,2013-07-09,"Very Positive 87% of the 841,501 user reviews ...","win, mac, linux",2026-10-14 00:26:00,steam_top_sellers_2026-10-14.csv,0.0,<NA>,<NA>
1,3393110,2,Action,AION 2,2026-10-05,"Mixed 54% of the 14,022 user reviews for this ...",win,2026-10-14 00:26:00,steam_top_sellers_2026-10-14.csv,0.0,<NA>,<NA>
2,730,3,Action,Counter-Strike 2,2012-08-21,"Very Positive 85% of the 2,632,865 user review...","win, linux",2026-10-14 00:26:00,steam_top_sellers_2026-10-14.csv,0.0,<NA>,<NA>
3,2828710,4,Action,Valor Mortis,2026-10-13,Very Positive 83% of the 67 user reviews for t...,win,2026-10-14 00:26:00,steam_top_sellers_2026-10-14.csv,38.23,44.98,15.01
4,2807960,5,Action,Battlefield™ 6,2025-10-10,"Mixed 65% of the 156,196 user reviews for this...",win,2026-10-14 00:26:00,steam_top_sellers_2026-10-14.csv,27.99,69.99,60.01


In [10]:

# Save the cleaned dataset with the correct scraping dates and times.
output_file = "steam_1_sellers_merged_cleaned.csv"

cleaned.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Corrected CSV saved successfully:", output_file)
print("Total records saved:", len(cleaned))
print("Missing scraping dates:", cleaned["scraped_at"].isna().sum())



Corrected CSV saved successfully: steam_1_sellers_merged_cleaned.csv
Total records saved: 3377
Missing scraping dates: 0


## Cell 10: Save the cleaned CSV backup

In [ ]:
OUTPUT_CSV = Path("steam_top_sellers_merged_cleaned.csv")
cleaned.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d %H:%M:%S")
print("Saved:", OUTPUT_CSV.resolve())

Saved: C:\Users\User\OneDrive\Desktop\steam_merge_mysql_project\steam_top_sellers_merged_cleaned.csv


## Cell 11: Connect to MySQL and create the database

In [ ]:

from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# MySQL connection settings
MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = getpass("Enter your MySQL password: ")

# Database and table names
DATABASE_NAME = "steam_data"
TABLE_NAME = "steam_top_sellers_cleaned"

# Create a connection to the MySQL server
server_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
)

server_engine = create_engine(server_url)

# Create the database if it does not exist
with server_engine.begin() as connection:
    connection.execute(
        text(f"CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}`")
    )

print(f"Database '{DATABASE_NAME}' is ready.")



## Cell 12: Upload the cleaned data to MySQL

In [ ]:
database_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=DATABASE_NAME,
)
db_engine = create_engine(database_url)

# if_exists='replace' replaces this table each time the notebook is rerun.
# Use this only if you want to refresh the table with the latest cleaned dataset.
cleaned.to_sql(
    name=TABLE_NAME,
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi",
)

print(f"Uploaded {len(cleaned):,} rows to {DATABASE_NAME}.{TABLE_NAME}")

## Cell 13: Verify the MySQL table

In [ ]:
with db_engine.connect() as connection:
    mysql_count = connection.execute(
        text(f"SELECT COUNT(*) FROM `{TABLE_NAME}`")
    ).scalar_one()
    sample_rows = pd.read_sql(
        text(f"SELECT * FROM `{TABLE_NAME}` LIMIT 5"), connection
    )

print("Rows in cleaned pandas dataset:", len(cleaned))
print("Rows in MySQL table:", mysql_count)
assert mysql_count == len(cleaned), "Row count mismatch: check the import."
display(sample_rows)

## MySQL Workbench verification

After the notebook runs successfully, open MySQL Workbench and execute:

```sql
USE steam_data;

SELECT COUNT(*) FROM steam_top_sellers_cleaned;

SELECT *
FROM steam_top_sellers_cleaned
LIMIT 10;
```

**Security:** The notebook asks for the MySQL password at runtime; do not save your password in the notebook. The import replaces the target table when rerun, so keep the cleaned CSV backup.